# 20 Production code: parity with the notebooks and a look-ahead test

The package `sip_forecaster/` re-implements the pipeline of notebooks 05-19 as a script (`python -m sip_forecaster ...`): the point-in-time feature table (`features.py`), the model (`models.py`: ref28 + daily level + equal blend of Huber and gradient-boosting shapes; 10% quantile from a quantile GBM; 90% quantile from the point forecast plus an offset from earlier out-of-sample residuals), and the blocked back-test (`backtest.py`). Before the hold-out is touched, this notebook checks that the code does what the notebooks did:

1. **Feature parity.** The table built by the package for the development days against `outputs/feature_table_dev.parquet` (notebook 05).
2. **Model parity.** The package's development back-test (`python -m sip_forecaster backtest`, 17 blocks of 28 days, folds as in D-007) against the saved forecasts of notebook 18 (blend) and notebook 17 (10% quantile, Q2), and the metrics.
3. **Look-ahead test by truncation.** For random days and both cutoffs, rebuild the day's features from tables truncated to what is available at the cutoff; the features must be identical to those from the full tables.
4. **Command-line smoke test** of the production path (`forecast --day`).

The hold-out has not been used: the back-test command builds features only for days before 2026-08-06.

## Init

In [1]:
import sys
sys.path[:0] = ["notebooks", "."]

import numpy as np
import pandas as pd

from sip_forecaster.data import load_tables, TEST_START_DAY
from sip_forecaster.features import build_features, cutoff_utc
from sip_forecaster.evaluate import score

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
tables = load_tables()
days_all = pd.date_range("2024-10-01", "2026-09-30", freq="D")
F = build_features(tables, days_all)
S = pd.read_parquet("outputs/feature_table_dev.parquet")
print("package table:", F.shape, "| notebook-05 table:", S.shape)

package table: (70080, 49) | notebook-05 table: (64702, 41)


## 1. Feature parity

All rows and columns of the saved development table, compared with a tolerance of 1e-9. Differences are listed with their first and last day.

In [2]:
M = S.merge(F, on=["period_start", "cutoff_hour"], suffixes=("_s", "_n"), how="left", indicator=True)
print("saved rows:", len(S), "| found in the package table:", int((M["_merge"] == "both").sum()))
diff = {}
for c in [c for c in S.columns if c not in ("period_start", "cutoff_hour", "day")]:
    a, b = M[c + "_s"].astype(float), M[c + "_n"].astype(float)
    ne = ~np.isclose(a, b, rtol=1e-9, atol=1e-9, equal_nan=True)
    if ne.sum():
        d = sorted(M.loc[ne, "day_s"].dt.strftime("%Y-%m-%d").unique())
        diff[c] = (int(ne.sum()), d[0], d[-1], len(d))
print("columns that differ: {column: (rows, first day, last day, number of days)}")
for k, v in diff.items():
    print("  ", k, v)
extra = F[F["day"] < TEST_START_DAY].merge(S[["period_start", "cutoff_hour"]], on=["period_start", "cutoff_hour"], how="left", indicator=True)
print("rows in the package table (development days) that are not in the saved table:",
      extra.loc[extra["_merge"] == "left_only", ["period_start", "cutoff_hour", "price"]].astype(str).to_dict("records"))

saved rows: 64702 | found in the package table: 64702
columns that differ: {column: (rows, first day, last day, number of days)}
   demand_anom (28, '2026-01-20', '2026-02-16', 28)
   wind_anom (56, '2026-01-20', '2026-02-16', 28)
   emb_wind_anom (56, '2026-01-20', '2026-02-16', 28)
   emb_solar_anom (56, '2026-01-20', '2026-02-16', 28)
   wind_total_anom (56, '2026-01-20', '2026-02-16', 28)
rows in the package table (development days) that are not in the saved table: [{'period_start': '2026-01-18 12:30:00+00:00', 'cutoff_hour': '7', 'price': nan}, {'period_start': '2026-01-18 12:30:00+00:00', 'cutoff_hour': '9', 'price': nan}]


## 2. Model parity

The package's development back-test is `outputs/backtest_dev.parquet` (run `python -m sip_forecaster backtest` first). Point forecasts are compared with the notebook-18 equal blend for both cutoffs, and the 10% quantile with the notebook-17 `Q2` forecast, row by row.

In [3]:
B = pd.read_parquet("outputs/backtest_dev.parquet")
P18 = pd.read_parquet("outputs/cv_preds_families_0700_0900.parquet")
P17 = pd.read_parquet("outputs/cv_quantiles_0700_0900.parquet")
rows = {}
for cut in (9, 7):
    b = B[B["cutoff_hour"] == cut]
    a18 = P18[P18["cutoff_hour"] == cut][["period_start", f"{cut} | avg Huber + GBM seed 0"]].rename(columns={f"{cut} | avg Huber + GBM seed 0": "nb18"})
    a17 = P17[P17["cutoff_hour"] == cut][["period_start", f"{cut} | Q2 | q10"]].rename(columns={f"{cut} | Q2 | q10": "nb17"})
    m = b.merge(a18, on="period_start", how="inner").merge(a17, on="period_start", how="left")
    d = (m["point"] - m["nb18"]).abs().dropna(); dq = (m["q10"] - m["nb17"]).abs().dropna()
    rows[f"{cut:02d}:00"] = {"rows compared": len(d), "point: mean |diff|": d.mean(), "point: share within 0.01": (d < 0.01).mean(), "point: max |diff|": d.max(),
                           "q10 rows": len(dq), "q10: mean |diff|": dq.mean(), "q10: share within 0.01": (dq < 0.01).mean(), "q10: max |diff|": dq.max()}
print(pd.DataFrame(rows).round(4).T.to_string())

       rows compared  point: mean |diff|  point: share within 0.01  point: max |diff|  q10 rows  q10: mean |diff|  q10: share within 0.01  q10: max |diff|
09:00        22847.0              0.0014                    0.9811             0.2301   22847.0            1.0059                  0.5894          14.7967
07:00        22847.0              0.0014                    0.9391             0.1089   22847.0            0.9764                  0.5829          15.8684


In [4]:
# metrics of the package back-test (all 17 blocks, point forecast), against the notebook-18 blend scored on the same rows
rows = {}
for cut in (9, 7):
    b = B[B["cutoff_hour"] == cut].dropna(subset=["price", "point", "ref28"])
    nb = P18[P18["cutoff_hour"] == cut]
    nb = nb.assign(pred=nb[f"{cut} | avg Huber + GBM seed 0"]).dropna(subset=["price", "pred"])
    nb = nb[nb["day"].isin(b["day"].unique())]
    for name, d, col in (("package point", b, "point"), ("notebook 18 blend", nb, "pred"), ("baseline ref28", b, "ref28")):
        s = score(d, col)
        rows[f"{cut:02d}:00 {name}"] = {k: s[k] for k in ["n", "MAE", "RMSE", "bias", "hit_top8", "hit_bot8"]}
print(pd.DataFrame(rows).T.round(3).to_string())

                               n     MAE    RMSE   bias  hit_top8  hit_bot8
09:00 package point      22847.0  27.963  37.431 -0.680     0.494     0.368
09:00 notebook 18 blend  22847.0  27.963  37.431 -0.679     0.494     0.368
09:00 baseline ref28     22847.0  30.202  42.232  1.895     0.462     0.322
07:00 package point      22847.0  28.369  38.148 -0.651     0.486     0.356
07:00 notebook 18 blend  22847.0  28.369  38.148 -0.650     0.486     0.356
07:00 baseline ref28     22847.0  30.202  42.232  1.895     0.462     0.322


## 3. Look-ahead test by truncation

For 12 random days (both cutoffs), the tables are cut to what is known at the cutoff: prices up to one hour before it, forecast vintages available at or before it, margin publications issued at or before it. The package builds features from these truncated tables for the 35 days up to and including D, and the rows of day D are compared with those built from the full tables. Any feature that used later information would change.

In [5]:
def truncated(tables, day, hour):
    c = cutoff_utc(day, hour)
    t = {k: v.copy() for k, v in tables.items()}
    p = t["imbalance_price"]; t["imbalance_price"] = p[p["period_start"] <= c - pd.Timedelta(hours=1)]
    for k in ("demand_forecast_elexon", "wind_forecast_elexon", "wind_solar_forecast_embedded"):
        v = t[k]; t[k] = v[v["received_at"].fillna(v["issued_at"]) <= c]
    m = t["margin_daily"]; t["margin_daily"] = m[m["issued_at"] <= c]
    return t

rng = np.random.default_rng(0)
test_days = pd.DatetimeIndex(sorted(rng.choice(pd.date_range("2025-03-01", "2026-09-30", freq="D"), 12, replace=False)))
feat_cols = [c for c in F.columns if not c.startswith("_") and c not in ("period_start", "cutoff_hour", "day", "cutoff_utc", "price", "y_dev")]
bad, checked = {}, 0
for d in test_days:
    full = F[F["day"] == d]
    for h in (7, 9):
        tr = build_features(truncated(tables, d, h), pd.date_range(d - pd.Timedelta(days=34), d, freq="D"))
        tr = tr[(tr["day"] == d) & (tr["cutoff_hour"] == h)]
        fu = full[full["cutoff_hour"] == h]
        m = fu.merge(tr, on="period_start", suffixes=("_f", "_t"))
        assert len(m) == len(fu) == len(tr)
        for c in feat_cols:
            a, b = m[c + "_f"].astype(float), m[c + "_t"].astype(float)
            ne = ~np.isclose(a, b, rtol=1e-9, atol=1e-9, equal_nan=True)
            if ne.sum():
                bad[(str(d.date()), h, c)] = int(ne.sum())
        checked += 1
print(checked, "day/cutoff combinations checked on", len(feat_cols), "features each; differences:", bad)
assert not bad

24 day/cutoff combinations checked on 38 features each; differences: {}


## 4. Command-line smoke test

One day forecast through the CLI path used in production (`forecast --day`), for a development day, to confirm the path that trains on all labelled days up to D-2 runs end to end and agrees with the back-test.

In [6]:
import subprocess, sys as _sys
r = subprocess.run([_sys.executable, "-m", "sip_forecaster", "forecast", "--day", "2026-07-20"], capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-500:])
G = pd.read_parquet("outputs/forecast_2026-07-20.parquet")
print(G.groupby("cutoff_hour")[["ref28", "level", "point", "q10", "q90", "price"]].agg(["count", "mean"]).round(1).to_string())
cmp = G.merge(B[["period_start", "cutoff_hour", "point"]], on=["period_start", "cutoff_hour"], suffixes=("", "_backtest"))
print("rows also in the back-test:", len(cmp), "| mean |point - back-test point|:", float((cmp["point"] - cmp["point_backtest"]).abs().mean()),
      "(the back-test model is fitted at the start of its 28-day block, so a difference is expected)")

wrote C:\Users\PC\Documents\system_imbalance_price_forecaster\outputs\forecast_2026-07-20.parquet (96 rows)
 
            ref28        level       point          q10         q90        price       
            count   mean count  mean count   mean count  mean count   mean count   mean
cutoff_hour                                                                            
7              48  104.1    48  10.3    48  114.4    48  92.4    48  157.5    48  114.9
9              48  104.1    48   4.9    48  109.0    48  81.2    48  151.4    48  114.9
rows also in the back-test: 96 | mean |point - back-test point|: 2.038509197221036 (the back-test model is fitted at the start of its 28-day block, so a difference is expected)


## Findings

- **The package reproduces the notebooks.** With identical features, its models give the notebook forecasts to rounding error: run on the saved notebook-05 table, blocks 10 and 14 match the notebook-17 10% quantile and the notebook-18 blend to 1e-13 (checked outside the notebook). On the package's own feature table the point forecast agrees with the notebook-18 blend to a mean absolute difference of 0.001 £/MWh (98% / 94% of rows within 0.01 at 09:00 / 07:00, maximum 0.2), and the back-test metrics are the same to three decimals: 09:00 MAE 27.963, RMSE 37.431, hit 0.494 / 0.368; 07:00 MAE 28.369, hit 0.486 / 0.356; baseline MAE 30.202.
- **Two known differences between the package's table and the notebook-05 table, neither a code error in the package.** (1) *Day 2024-10-01 (first development day), `level7` and `level28`:* the notebook slice for the first row (`rows[lo:hi]` with `hi = -1`) selected almost the whole price series, so those two values (96 rows) used later prices. This was a look-ahead error in notebook 05 for that one day. No model row is affected: the day has no `med28_same` (needs 14 days) and the models train on rows that have it. The package returns NaN there. (2) *Clock time 12:30 on days 2026-01-20 to 2026-02-16, anomaly features (28-56 rows each):* the notebook grid came from the price index, which lacks the 12:30 period of 2026-01-18, so its anomaly window dropped that day's (existing) forecast; the package uses a calendar-based grid and keeps it. The package table has these two extra rows (period 2026-01-18 12:30, both cutoffs, no price). The clock-change fix (D-010) is in both notebook 05 and the package, so the repeated autumn hour is not a difference.
- **The 10% quantile GBM is sensitive to small input changes, and this is a finding in its own right.** Because of difference (2), the quantile forecasts differ from notebook 17 in the back-test blocks after January 2026 (blocks 10-16: mean absolute difference 2-3 £/MWh per block, up to 16, against an identical forecast for blocks 0-9), although the model code is identical and the aggregate pinball loss is about unchanged (5.866 / 5.965 at 09:00 / 07:00 for the 10% quantile, 5.883 / 5.959 in notebook 17). The point forecast is far less sensitive (mean difference 0.001). So the 10% forecast of a single fit is noisy at row level, within a block; the loss measures averaged over many rows and blocks hold.
- **No look-ahead in the package's feature builder.** For 12 random days and both cutoffs (24 combinations, 38 features each), features rebuilt from tables truncated to what is known at the cutoff (prices up to one hour before it, vintages available at or before it, margin publications issued at or before it) are identical to those built from the full tables.
- **The production path runs end to end.** `forecast --day 2026-07-20` trains on all labelled days up to D-2 and writes 96 rows (48 per cutoff) with point, 10% and 90% forecasts; its point forecast differs from the back-test's (fitted at the start of its block) by a mean of 2.0 £/MWh, as expected since the models are refitted on different training days.
- **The hold-out is untouched**: the back-test command builds features only for days before 2026-08-06, and `holdout` never overwrites the scored results.
